# Cancer + all 4 views (READ-ONLY)

Count cancer studies with `l-cc`, `r-cc`, `l-mlo`, `r-mlo`.
Only reads `/raid`.

In [8]:
import pydicom
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [9]:
NEED = {"l-cc", "r-cc", "l-mlo", "r-mlo"}

In [10]:
def get_view(ds):
    v = str(ds.get("ViewPosition") or "").lower()
    if not v and getattr(ds, "ViewCodeSequence", None):
        v = str(ds.ViewCodeSequence[0].CodeMeaning).lower()

    if "cc" in v or "cranio" in v:
        return "cc"
    if "mlo" in v or "oblique" in v or "medio" in v:
        return "mlo"
    return ""

In [11]:
def cancer_four(folder, label_csv):
    df = pd.read_csv(label_csv)
    cancer_ids = set(
        df[df["Label"].isin(["IndexCancer", "PreIndexCancer"])]["StudyInstanceUID"].astype(str)
    )

    views = defaultdict(set)

    for sid in cancer_ids:
        for f in Path(folder, sid).glob("DXm.*"):
            try:
                ds = pydicom.dcmread(str(f), stop_before_pixels=True, force=True)
            except Exception:
                continue

            lat = str(ds.get("ImageLaterality", "")).lower()
            v = get_view(ds)
            if lat in ("l", "r") and v:
                views[sid].add(f"{lat}-{v}")

    n_four = sum(NEED <= s for s in views.values())
    return n_four, len(views), len(cancer_ids)

In [12]:
# (cancer with 4 views, cancer studies on disk, cancer ids in CSV)
print(cancer_four(
    "/raid/data01/deephealth/dh_dcm_ast",
    "/raid/data01/deephealth/labels/dh_dcm_ast_labels.csv",
))

(2232, 4364, 4381)


In [13]:
print(cancer_four(
    "/raid/data01/deephealth/dh_dh0new",
    "/raid/data01/deephealth/labels/dh_dh0new_labels.csv",
))

(129, 159, 159)


In [14]:
print(cancer_four(
    "/raid/data01/deephealth/dh_dh2",
    "/raid/data01/deephealth/labels/dh_dh2_labels.csv",
))

(10, 13, 13)
